In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:

df = pd.read_csv('/kaggle/input/datasets/jessicali9530/celeba-dataset/list_attr_celeba.csv')


df.columns.tolist()

df[['Chubby', 'Black_Hair']]


,Chubby,Black_Hair
0,-1,-1
1,-1,-1
2,-1,-1
3,-1,-1
4,-1,-1
...,...,...
202594,-1,-1
202595,-1,-1
202596,-1,1
202597,-1,1


In [5]:
!pip install git+https://github.com/facebookresearch/ImageBind.git#egg=imagebind
!pip install setuptools<82
!pip install torchcodec

  Cloning https://github.com/facebookresearch/ImageBind.git to /tmp/pip-install-jnhnj_ty/imagebind_d7973c4fa0674878875faaaaf0449e4d
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/ImageBind.git /tmp/pip-install-jnhnj_ty/imagebind_d7973c4fa0674878875faaaaf0449e4d
  Resolved https://github.com/facebookresearch/ImageBind.git to commit 53680b02d7e37b19b124fa37bae4b6c98c38f5be
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/facebookresearch/pytorchvideo.git (to revision 6cdc929315aab1b5674b6dcf73b16ec99147735f) to /tmp/pip-install-jnhnj_ty/pytorchvideo_48426aa810a64bcab408534f0d913e35
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorchvideo.git /tmp/pip-install-jnhnj_ty/pytorchvideo_48426aa810a64bcab408534f0d913e35
  Running command git rev-parse -q --verify '6cdc929315aab1b5674b6dcf73b16

In [6]:
import torch
from imagebind.models.imagebind_model import ImageBindModel
from imagebind.models import imagebind_model
from imagebind import data
from imagebind.models.imagebind_model import ModalityType

/usr/local/lib/python3.13/dist-packages/imagebind/data.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [7]:
device = "cuda:0" # "cpu", "cuda:0"

model = imagebind_model.imagebind_huge(pretrained=True)

torch.cuda.get_device_name(0)

100%|██████████| 4.47G/4.47G [01:42<00:00, 47.1MB/s]


'Tesla T4'

In [8]:
model.eval()
model.to(device)

ImageBindModel(
  (modality_preprocessors): ModuleDict(
    (vision): RGBDTPreprocessor(
      (cls_token): tensor((1, 1, 1280), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Sequential(
          (0): PadIm2Video()
          (1): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
        )
      )
      (pos_embedding_helper): SpatioTemporalPosEmbeddingHelper(
        (pos_embed): tensor((1, 257, 1280), requires_grad=True)

      )
    )
    (text): TextPreprocessor(
      (pos_embed): tensor((1, 77, 1024), requires_grad=True)
      (mask): tensor((77, 77), requires_grad=False)

      (token_embedding): Embedding(49408, 1024)
    )
    (audio): AudioPreprocessor(
      (cls_token): tensor((1, 1, 768), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10), bias=False)
        (norm_layer): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (pos_e

In [19]:
sampled_df = df.sample(n=5000)
images = '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba/'+ sampled_df['image_id']
metadata = sampled_df[['Chubby', 'Black_Hair']]

In [10]:
batch_size = 500

round = 0
all_embeddings = []
for i in range(0, len(images), batch_size):
    print("Round", round)
    cur_input = {
        ModalityType.VISION: data.load_and_transform_vision_data(images[i:i+batch_size], device)
    }
    print("input loaded")

    with torch.amp.autocast(device, dtype=torch.float16):
        with torch.no_grad():
            embeddings = model(cur_input)
    
    print(embeddings)
    print()
    all_embeddings.append(embeddings)
    round += 1



Round 0
input loaded
{'vision': tensor([[-3.3342e-04,  4.4261e-03, -2.2312e-02,  ..., -1.5219e-02,
          3.0907e-02, -2.9922e-02],
        [ 1.4785e-02, -6.0284e-02, -1.7616e-02,  ...,  4.3723e-04,
          2.3356e-03,  1.3110e-02],
        [ 2.5304e-02, -2.9521e-02,  1.1236e-02,  ..., -3.1924e-02,
          2.5966e-02, -1.9640e-02],
        ...,
        [-7.7240e-03, -1.2510e-02, -3.5651e-02,  ...,  3.4139e-02,
         -4.8130e-03, -9.5873e-05],
        [ 1.7400e-02, -3.4294e-03, -1.9713e-02,  ...,  4.4997e-02,
          1.8804e-02, -5.9244e-02],
        [-5.3603e-03, -2.4635e-02, -6.8942e-03,  ..., -2.5843e-02,
         -9.1296e-03, -1.0396e-02]], device='cuda:0')}

Round 1
input loaded
{'vision': tensor([[-0.0206, -0.0142,  0.0186,  ...,  0.0290,  0.0387,  0.0063],
        [-0.0030,  0.0112, -0.0116,  ...,  0.0759,  0.0388, -0.0298],
        [ 0.0084, -0.0339,  0.0141,  ..., -0.0318,  0.0049, -0.0030],
        ...,
        [-0.0326,  0.0142, -0.0247,  ...,  0.0353,  0.0420,  0

In [11]:
!pip install lancedb

In [15]:
all_embeddings[0]['vision'][0:2].shape

torch.Size([2, 1024])

In [32]:
to_ingest = []

round = 0
for i in range(0, len(images), batch_size):
    print("Round", round)
    data_part = sampled_df[['image_id','Chubby', 'Black_Hair']].iloc[i : i + batch_size].to_dict(orient="list")
    vals = list(data_part.values()) + [all_embeddings[round]['vision'].tolist()]
    keys = list(data_part.keys()) + ['vector']
        
    data_ingest = [dict(zip(keys, values)) for values in zip(*vals)]

    to_ingest += data_ingest
    round += 1

print(len(to_ingest))
to_ingest[0]


Round 0
Round 1
Round 2
Round 3
Round 4
Round 5
Round 6
Round 7
Round 8
Round 9
5000


{'image_id': '056161.jpg',
 'Chubby': -1,
 'Black_Hair': -1,
 'vector': [-0.0003334239881951362,
  0.004426111932843924,
  -0.022312292829155922,
  0.0004590991884469986,
  -0.0004224591830279678,
  0.053230591118335724,
  -0.02992168813943863,
  -0.03318118304014206,
  -0.01621539704501629,
  -0.04504667967557907,
  -0.025630412623286247,
  -0.031914904713630676,
  0.01221724133938551,
  -0.006782796699553728,
  0.04999454692006111,
  0.027858123183250427,
  0.006565887946635485,
  -0.028843006119132042,
  -0.00434403819963336,
  -0.02312130481004715,
  -0.11058831214904785,
  0.06664375960826874,
  -0.007978726178407669,
  -0.0071169533766806126,
  -0.029992036521434784,
  0.013858713209629059,
  0.0559038445353508,
  0.006823833100497723,
  -0.002850591903552413,
  -0.015113266184926033,
  0.03995811566710472,
  0.04202168062329292,
  0.01031196117401123,
  -0.02022527903318405,
  -0.026709092780947685,
  0.02011975646018982,
  -0.01494911964982748,
  0.01110924780368805,
  -0.02359

In [21]:
import lancedb

uri = 'celeba_db'
db = lancedb.connect(uri)

In [23]:
table = db.create_table("celeba", data=to_ingest, mode="overwrite")

In [37]:
query_vector = all_embeddings[0]['vision'][0].tolist()


result = (
    table.search(query_vector, vector_column_name="vector")
    .select(["image_id", "_distance"])
    .limit(10)
    .to_polars()
)
print(result)




shape: (10, 2)
┌────────────┬───────────┐
│ image_id   ┆ _distance │
│ ---        ┆ ---       │
│ str        ┆ f32       │
╞════════════╪═══════════╡
│ 056161.jpg ┆ 0.0       │
│ 194945.jpg ┆ 0.420473  │
│ 012139.jpg ┆ 0.442851  │
│ 075914.jpg ┆ 0.49102   │
│ 136203.jpg ┆ 0.491575  │
│ 129414.jpg ┆ 0.495651  │
│ 128827.jpg ┆ 0.498128  │
│ 039092.jpg ┆ 0.511295  │
│ 086750.jpg ┆ 0.516124  │
│ 029903.jpg ┆ 0.526532  │
└────────────┴───────────┘
